<a name="Notebook-Start"></a>

---

<font size = 7> <b> Lesson 2.4 (Avoiding Overfit Models) </b> </font>

---

<font size = 5> <b> Notebook Index </b> </font>

1.  [Learning Outcomes](#Learning-Outcomes)

2.  [Introduction](#Introduction)

3.  [Import Python Libraries](#Import-Python-Libraries)

4.  [Define Useful Functions](#Define-Useful-Functions)

5.  [Load The Dataset](#Load-The-Dataset)

6.  [Recap PolynomialFeatures Transformer](#PFT)

7.  [Cross Validation](#CV)

8.  [Introducing Pipelines](#Pipelines)
$
\newcommand{\parens}[1]{\left( #1 \right)}
\newcommand{\brackets}[1]{\left[ #1 \right]}
\newcommand{\lsum}{\displaystyle \sum\limits_{i=1}^{N}}
\newcommand{\parens}[1]{\left(#1\right)}
\newcommand{\dsfrac}[2]{\displaystyle\frac{#1}{#2}}
\newcommand{\dpfrac}[2]{\displaystyle\parens{\frac{#1}{#2}}}
\newcommand{\parderiv}[2]{\dsfrac{\partial #1}{\partial #2}}
\newcommand{\spc}{\hspace{0.1 pc}}
\newcommand{\ra}{\Rightarrow}
\newcommand{\of}[1]{{\scriptsize (#1)}}
\newcommand{\rule}{\Huge \hspace{-0.2 pc} \displaystyle\frac{\hspace{20 pc}}{\hspace{20 pc}}}
\newcommand{\mps}{\spc \frac{\textrm{m}}{\textrm{s}}}
\newcommand{\mpss}{\spc \frac{\textrm{m}}{\textrm{s}^2}}
\newcommand{\twomatrixtall}[2]{
\left( \begin{array}{c} #1 \\ #2 \end{array} \right)
}
\newcommand{\threematrixtall}[3]{
\left( \begin{array}{c} #1 \\ #2 \\ #3 \end{array} \right)
}
\newcommand{\fourmatrixsquare}[4]{
\left( \begin{array}{c c} #1 & #2 \\ #3 & #4 \end{array} \right)
}
\newcommand{\fourmatrixlong}[4]{
\left( \begin{array}{c c c c} #1 & #2 & #3 & #4 \end{array} \right)
}
\newcommand{\ninematrixsquare}[9]{
\left( \begin{array}{c c c} #1 & #2 & #3 \\ #4 & #5 & #6 \\ #7 & #8 & #9 \end{array} \right)
}
$

<a name="Learning-Outcomes"></a>

---

<font size = 6> <b> 1. Learning Outcomes </b> </font>

---

<font size = 5> <b> Learning Outcomes: </b> </font>

By the end of this lesson, students will be able to:

  1.

[Return to Top](#Notebook-Start)

<a name="Introduction"></a>

---

<font size = 6> <b> 2. Introduction </b> </font>

---

Overfitting is a type of error that occurs when a model is too closely aligned to a training dataset. Consequently, the model is valid only in relation to its training dataset and not any subsequent dataset. Overfitting is a common problem in model building, but it typically can be handled by utilizing the following techniques: holdout (test/train split), cross-validation, feature selection (engineering), and data augmentation.

[Return to Top](#Notebook-Start)

<a name="Import-Python-Libraries"></a>

---

<font size = 6> <b> 3. Import Python Libraries </b> </font>

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display_html
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

[Return to Top](#Notebook-Start)

<a name="Define-Useful-Functions"></a>

---

<font size = 6> <b> 4. Define Useful Functions </b> </font>

---

In [ ]:
#@title This cell defines the functions: display_dataframes, load_model, display_models, and plot_data

##=============================================================================================##
## Included Functions:                                                                         ##
##                                                                                             ##
## 1. display_dataframes - Display multiple DataFrames side-by-side with titles                ##
## 2. load_model         - Load and clean data from input file, split into feature and target  ##
## 3. display_models      - Display models' parameters and loss in a DataFrame                 ##
## 4. plot_data          - Create a graph with raw data, can add model to graph if needed      ##
##=============================================================================================##

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  dataframe_list - List of DataFrames to be displayed                              ##
##            title_list     - List of titles for the displayed DataFrames                     ##
##            n_items        - Number of items to display (optional, default = 5)              ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(dataframe_list, title_list, n_items = 5):

  ##===========================================================================================##
  ## Create a String for Housing the Commands to Be Sent to the display_html() Function:       ##
  ##===========================================================================================##

  html_str = ""

  ##===========================================================================================##
  ## Loop Over the Elements in the item_list and title_list:                                   ##
  ##===========================================================================================##

  for df, title in zip(dataframe_list, title_list):

    # Convert the current dataframe info to html:

    html_df = pd.DataFrame(df).head(n_items).to_html()

    ##=========================================================================================##
    ## Wrap title and DataFrames in a Styled HTML <div>:                                       ##
    ##=========================================================================================##

    html_str += "<div style='display: inline-block; margin-right: 20px; vertical-align: top;'>"

    html_str += "<h3 style='text-align: center;'>" + str(title) + "</h3><hr>" + str(html_df)

    html_str += "</div>"

  ##===========================================================================================##
  ## Send the HTML String to the display_html() Function:                                      ##
  ##===========================================================================================##

  display_html(html_str, raw = True)

##=============================================================================================##
## Function:  load_model                                                                       ##
##                                                                                             ##
## Purpose:   Load and clean data from input file, split into feature and target               ##
##                                                                                             ##
## Input(s):  filename     - Name of the file containing the data                              ##
##            feature_list - List of column names containing feature data                      ##
##            target_list  - List of column names containing target data                       ##
##                                                                                             ##
## Output(s): features     - DataFrame containing model feature data                           ##
##            targets      - DataFrame containint model target data                            ##
##=============================================================================================##

def load_model(filename, feature_list, target_list):

  ##===========================================================================================##
  ## Load in the Full Data Set and Drop Any Rows Missing Data:                                 ##
  ##===========================================================================================##

  data = pd.read_csv(filename).dropna()

  ##===========================================================================================##
  ## Separate the Feature and Target Data and Drop Any Rows Missing Data:                      ##
  ##===========================================================================================##

  # Identify the feature data:

  features = pd.DataFrame(data[feature_list])

  # Identify the target data:

  targets = pd.DataFrame(data[target_list])

  # Display the feature and target data:

  display_dataframes([data, features, targets], ["Full Dataset", "Feature Data", "Target Data"])

  ##===========================================================================================##
  ## Return the Feature Data, and Target Data:                                                 ##
  ##===========================================================================================##

  return features, targets

##=============================================================================================##
## Function:  display_models                                                                   ##
##                                                                                             ##
## Purpose:   Display models' parameters and loss in a DataFrame                               ##
##                                                                                             ##
## Input(s):  model_list - List of models' names to be displayed                               ##
##            coef_list  - List of models' coefficients to be displayed                        ##
##            bias_list  - List of models' bias to be displayed                                ##
##            loss_list  - List of models' loss to be displayed                                ##
##            title      - Title for the display of models                                     ##
##            trunc      - Number of decimals to display for numbers (optional, default = 3)   ##
##            n_items    - Number of items to display (optional, default = 5)                  ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_model(model_list, coef_list, bias_list, loss_list, title, trunc = 3, n_items = 5):

  ##===========================================================================================##
  ## Round the Numeric Values to the Desired Level of Desired Truncation:                      ##
  ##===========================================================================================##

  for i in range (0, len(model_list)):

    coef_list[i] = np.round(coef_list[i], trunc)
    bias_list[i] = np.round(bias_list[i], trunc)
    loss_list[i] = np.round(loss_list[i], trunc)

  ##===========================================================================================##
  ## Create a DataFrame to Hold the Results:                                                   ##
  ##===========================================================================================##

  results = pd.DataFrame()

  ##===========================================================================================##
  ## Add the Contents of the DataFrame Columns:                                                ##
  ##===========================================================================================##

  # Add the model names:

  results["Model"] = model_list

  # Add the model coefficients:

  results["Coefficient(s)"] = coef_list

  # Add the model biases:

  results["Bias"] = bias_list

  # Add the model rmses:

  results["Loss"] = loss_list

  ##===========================================================================================##
  ## Index the Results DataFrame By Model Name:                                                ##
  ##===========================================================================================##

  results.set_index("Model", inplace = True)

  ##===========================================================================================##
  ## Display the Results DataFrame Using display_dataframes():                                 ##
  ##===========================================================================================##

  display_dataframes([results], [title], n_items)

##=============================================================================================##
## Function:  plot_data                                                                        ##
##                                                                                             ##
## Purpose:   Create a scatterplot with optional model overlays and error bands                ##
##                                                                                             ##
## Input(s):  x_data        - List of data points' x-axis values                               ##
##            y_data        - List of data points' y-axis values                               ##
##            title         - Graph title                                                      ##
##            axis_labels   - Override axis labels [x_label, y_label] (optional)               ##
##            model_list    - List of model predictions to overlay (optional)                  ##
##            color_list    - Colors for each model line (optional)                            ##
##            label_list    - Labels for each model line (optional)                            ##
##            error_display - Show +/- error band around first model (default is False)        ##
##            error         - Error value for shaded band (optional)                           ##
##                                                                                             ##
## Output(s): graph       - Matplotlib axes object, can be used for overplotting               ##
##=============================================================================================##

def plot_data(x_data, y_data, title, axis_labels = [], model_list = [], color_list = [],
              label_list = [], error_display = False, error = 0):

  ##===========================================================================================##
  ## Setup the Graph:                                                                          ##
  ##===========================================================================================##

  # Create the Matplotlib figure:

  figure = plt.figure(figsize = (12, 9))

  # Add a graph to the figure:

  graph = figure.add_subplot()

  # Set the graph background Color:

  graph.set_facecolor('lightcyan')

  # Set the graph title:

  graph.set_title(title, fontsize = 20)

  # Set the x_label and y_label:

  if (axis_labels != []):

    graph.set_xlabel(axis_labels[0], fontsize = 14)

    graph.set_ylabel(axis_labels[1], fontsize = 14)

  # Apply a grid to the graph:

  graph.grid(which = 'both')

  # Adjust the x-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'x', tight = False)

  # Adjust the y-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'y', tight = False)

  ##===========================================================================================##
  ## Add Data to the Graph:                                                                    ##
  ##===========================================================================================##

  # Create a scatterplot of the data:

  sns.scatterplot(x = x_data, y = y_data, ax = graph)

  # Overlay model predictions:

  for i in range(0, len(model_list)):

    graph.plot(model_list[i]["Target"], model_list[i]["Predictions"], color = color_list[i],
               label = label_list[i])

  ##===========================================================================================##
  ## If requested, show the +/- error bounds:                                                  ##
  ##===========================================================================================##

  if ((error_display == True) and model_list != []):

    # Create the error+ model:

    model_plus_error  = model_list[-1]["Predictions"] + error

    # Create the error- model:

    model_minus_error = model_list[-1]["Predictions"] - error

    # Store the error+ and error- models in a DataFrame and Sort by x_data values:

    error_df = pd.DataFrame({
        'X-Data': model_list[0]["Target"],
        'Model+Error': model_plus_error,
        'Model-Error': model_minus_error
    }).sort_values(by = "X-Data")

    # Use graph.fill to highlight the region between the error+ and error- models:

    graph.fill_between(error_df['X-Data'], error_df['Model+Error'], error_df['Model-Error'],
                       alpha = 0.5, color = (0.6, 0.6, 0.6), label = "Error Bounds")

  ##===========================================================================================##
  ## Apply the Legend and Return the graph Object:                                             ##
  ##===========================================================================================##

  # Add the graph legend:

  if (label_list != []): graph.legend()

  # Return the graph:

  return graph

[Return to Top](#Notebook-Start)

<a name="Load-The-Dataset"></a>

---

<font size = 6> <b> 5. Load The Dataset </b> </font>

---

<font size = 5> <b> 5.1 What Kind Of Data Are We Using? </b> </font>

We'll begin with the same synthetic dataset of a falling object that we used in the previous lesson. Recall that this dataset includes two key simplifications:

  <br>

  1. The object is falling <b>without air resistance</b>.

  <br>

  2. The object starts with <b>zero initial speed</b>.

  <br>

These simplifications were made to allow us to focus on introducing the modeling process without needing to deal with extra complications.

<center>$\rule$</center>

<font size = 5> <b> 5.2 What's In The Dataset? </b> </font>

Each row in the dataset represents a single measurement taken during the object's fall and includes:

  <br>

  * <b>Time (s)</b>: How many seconds have passed since the object started falling

  * <b>Speed (m/s)</b>: The object's speed at the moment of measurement, in meters per second

  * <b>Position (m)</b>: The object's fall distance at the moment of measurement, in meters.

<center>$\rule$</center>

<font size = 5> <b> 5.3 Why This Dataset? </b> </font>

We're starting with this dataset to build on what we learned in the previous lesson. It's simple, clean, and easy to interpret, and we can compare our results with what we got last time.

In [ ]:
##=============================================================================================##
## Load The Full Data Set:                                                                     ##
##=============================================================================================##

github = "https://raw.githubusercontent.com/dr-bankert-augustana/PHYS_200/refs/heads/main/"

url = github + "Data/Module_2/freefall_data_1.csv"

full_data = pd.read_csv(url)

##=============================================================================================##
## Clean the Data and Separate Features and Targets:                                           ##
##=============================================================================================##

# Remove any rows missing data:

cleaned_data = full_data.dropna()

# Identify the feature data:

X = pd.DataFrame(cleaned_data["Time (s)"])

# Identify the target data:

y = pd.DataFrame(cleaned_data["Position (m)"])

##=============================================================================================##
## Display the Full Dataset, Feature Data, and Target Data:                                    ##
##=============================================================================================##

# Display the feature and target data:

display_dataframes([full_data, X, y], ["Full Dataset", "Feature Data", "Target Data"])

print('\n\n')

##=============================================================================================##
## Create a Plot of the Data:                                                                  ##
##=============================================================================================##

# Set the title of the plot:

title = "Position vs Time"

# Set the x-axis and y-axis labels:

x_label = "Time (s)"
y_label = "Position (m)"

# Create a scatterplot of the feature data vs the target data:

graph = plot_data(X[x_label], y[y_label], title, [x_label, y_label])

[Return to Top](#Notebook-Start)

<a name="PFT"></a>

---

<font size = 6> <b> 6. Recap PolynomialFeatures Transformer</b> </font>

---

<font size = 5> <b> 6.1 Test the First N Polynomial Orders </b> </font>

In [ ]:
##=============================================================================================##
## Use PolynomialFeatures to generate a nth order model:                                       ##
##=============================================================================================##

n_poly = 10

##=============================================================================================##
## Create a List of Features:                                                                  ##
##=============================================================================================##

X_list = []

# Loop over polynomial orders from 1 to n_poly:

for i in range(1, n_poly + 1):

    # Create a PolynomialFeatures transformer:

    poly_transform = PolynomialFeatures(degree = i, include_bias = False)

    # Generate the Polynomial Features DataFrame:

    X_list.append(pd.DataFrame(poly_transform.fit_transform(X),
                               columns = poly_transform.get_feature_names_out()))

##=============================================================================================##
## Create Lists to Hold the Model Results:                                                     ##
##=============================================================================================##

# Create a List of model names based on polynomial order:

model_names = ["Linear", "Quadratic", "Cubic", "Quartic", "Quintic", "Sextic", "Septic",
               "Octic", "Nonic", "Decic"]

model_list = model_names[:n_poly]

# Create a List of coefficients:

coef_list = []

# Create a List of biases:

bias_list = []

# Create a List of losses:

loss_list = []

##=============================================================================================##
## Use a LinearRegression Object to Find the Best Fit for Each Polynomial Model:               ##
##=============================================================================================##

# Create a LinearRegression object with a forced-origin intercept:

model = LinearRegression(fit_intercept = False)

# Loop over polynomial orders from 1 to n_poly:

for i in range(0, n_poly):

  # Fit the LinearRegression objects to the features and target:

  model.fit(X_list[i], y)

  # Get the coefficient and bias for the model:

  coef_list.append(np.round(model.coef_[0], 2))
  bias_list.append(np.round(model.intercept_, 2))

  # Get the model's loss:

  model_predictions = model.predict(X_list[i])
  loss_list.append(np.sqrt(mean_squared_error(y, model_predictions)))

##=============================================================================================##
## Display The Results:                                                                        ##
##=============================================================================================##

# Show the results in DataFrame format:

display_model(model_list, coef_list, bias_list, loss_list, "Polynomial Models", n_items = n_poly,
              trunc = 5)

print('\n\n')

# Create a line plot of Loss vs Polynomial Order

loss_df = pd.DataFrame({"Model": model_list, "Loss": loss_list})

plt.figure(figsize=(10, 6))

sns.lineplot(x = 'Model', y = 'Loss', data=loss_df)

# Add title and labels

plt.title('RMSE Loss vs Model Order')
plt.xlabel('Model')
plt.ylabel('RMSE Loss')

# Set the plot parameters:

plt.grid(True)

y_min = loss_df["Loss"].min() * 0.995
y_max = loss_df["Loss"][1:].max() * 1.005

plt.ylim(y_min, y_max)

# Display the plot

plt.show()

[Return to Top](#Notebook-Start)

<a name="CV"></a>

---

<font size = 6> <b> 7. Cross Validation</b> </font>

---

<font size = 5> <b> 7.1 Train_Test_Split </b> </font>

In order to avoid overfitting our model, we will split the data before we analyzie it into two groups. The first group will be the <b>training data</b> from which we will build our model. The second group will be the <b>testing data</b> which we will use to test our model's accuracy. Because the model that we create has never seen the testing data, it cannot be overfit on it, thus it provides us an unbiased (pardon the pun) view of our model's ability to accurately capture the data trends. This is called <b>Cross Validation</b>

<br>

Fortunately for us, SKlearn has a built in function called <b>train_test_split</b> that will automatically separate the data for us. We just need to give it (Features, Targets, percent of the data to use for testing, and a number to seed the choosing of which data goes where).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

display_dataframes([X_train, y_train, X_test, y_test], ["Training Feature Data", "Training Target Data", "Testing Feature Data", "Testing Target Data"])


<center>$\rule$</center>

<font size = 5> <b> 7.2 Test the First N Polynomial Orders Again</b> </font>

Let's find the best polynomial order again, this time, we'll train the model on the training data and test it on the testing data.

In [ ]:
##=============================================================================================##
## Use PolynomialFeatures to generate a 10th order model:                                      ##
##=============================================================================================##

##=============================================================================================##
## Create a List of Features:                                                                  ##
##=============================================================================================##

X_list_train = []

X_list_test = []

# Loop over polynomial orders from 1 to 10:

for i in range(1, 11):

    # Create a PolynomialFeatures transformer:

    poly_transform = PolynomialFeatures(degree = i, include_bias = False)

    # Generate the Polynomial Features DataFrame:

    X_list_train.append(pd.DataFrame(poly_transform.fit_transform(X_train),
                               columns = poly_transform.get_feature_names_out()))

    X_list_test.append(pd.DataFrame(poly_transform.fit_transform(X_test),
                               columns = poly_transform.get_feature_names_out()))

##=============================================================================================##
## Create Lists to Hold the Model Results:                                                     ##
##=============================================================================================##

# Create a List of model names based on polynomial order:

model_list = ["Linear", "Quadratic", "Cubic", "Quartic", "Quintic", "Sextic", "Septic",
              "Octic", "Nonic", "Decic"]

# Create a List of coefficients:

coef_list = []

# Create a List of biases:

bias_list = []

# Create a List of losses:

loss_list = []

##=============================================================================================##
## Use a LinearRegression Object to Find the Best Fit for Each Polynomial Model:               ##
##=============================================================================================##

# Create a LinearRegression object with a forced-origin intercept:

model = LinearRegression(fit_intercept = False)

# Loop over polynomial orders from 1 to 10:

for i in range(0, 10):

  # Fit the LinearRegression objects to the features and target:

  model.fit(X_list_train[i], y_train)

  # Get the coefficient and bias for the model:

  coef_list.append(np.round(model.coef_[0], 2))
  bias_list.append(np.round(model.intercept_, 2))

  # Get the model's loss:

  model_predictions = model.predict(X_list_test[i])
  loss_list.append(np.sqrt(mean_squared_error(y_test, model_predictions)))

##=============================================================================================##
## Display The Results:                                                                        ##
##=============================================================================================##

# Show the results in DataFrame format:

display_model(model_list, coef_list, bias_list, loss_list, "Polynomial Models", n_items = n_poly,
              trunc = 5)

print('\n\n')

# Create a line plot of Loss vs Polynomial Order

loss_df = pd.DataFrame({"Model": model_list, "Loss": loss_list})

plt.figure(figsize=(10, 6))

sns.lineplot(x = 'Model', y = 'Loss', data=loss_df)

# Add title and labels

plt.title('RMSE Loss vs Model Order')
plt.xlabel('Model')
plt.ylabel('RMSE Loss')

# Set the plot parameters:

plt.grid(True)

y_min = loss_df["Loss"].min() * 0.995
y_max = loss_df["Loss"][1:].max() * 1.005

plt.ylim(y_min, y_max)

# Display the plot

plt.show()

We can see that the best order is 'Quadradic', which is what we expect from our knowledge of physics. One downside of this method, though, is that it is highly dependent on which data ends up in the training data and which ends up in the testing data. Go ahead and change the value in random_state = # in the train_test_split function to 90 and re-evaluate the best polynomial order.

<br>

<b>Uh Oh!</b> What happened, why did math and physics betray us!

<br>

You won't always get the best result by running this analysis once, because the split is completely random. How do we fix this? How do we know which is the best result? One quick and easy way is to run the analysis a bunch of times and find the average loss for each model.

In [ ]:
##=============================================================================================##
## Run the analyis N_Runs number of times and average the results:                             ##
##=============================================================================================##

n_poly = 10
n_runs = 100

loss_list_average = np.zeros(n_poly)

for j in range(0, n_runs):

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = int(np.random.rand()*100000))

  ##=============================================================================================##
  ## Use PolynomialFeatures to generate a 10th order model:                                      ##
  ##=============================================================================================##

  ##=============================================================================================##
  ## Create a List of Features:                                                                  ##
  ##=============================================================================================##

  X_list_train = []

  X_list_test = []

  # Loop over polynomial orders from 1 to n_poly:

  for i in range(1, n_poly + 1):

    # Create a PolynomialFeatures transformer:

    poly_transform = PolynomialFeatures(degree = i, include_bias = False)

    # Generate the Polynomial Features DataFrame:

    X_list_train.append(pd.DataFrame(poly_transform.fit_transform(X_train),
                               columns = poly_transform.get_feature_names_out()))

    X_list_test.append(pd.DataFrame(poly_transform.fit_transform(X_test),
                               columns = poly_transform.get_feature_names_out()))

  ##=============================================================================================##
  ## Create Lists to Hold the Model Results:                                                     ##
  ##=============================================================================================##

  # Create a List of model names based on polynomial order:

  model_names = ["Linear", "Quadratic", "Cubic", "Quartic", "Quintic", "Sextic", "Septic",
                "Octic", "Nonic", "Decic"]

  model_list = model_names[:n_poly]

  # Create a List of coefficients:

  coef_list = []

  # Create a List of biases:

  bias_list = []

  # Create a List of losses:

  loss_list = []

  ##=============================================================================================##
  ## Use a LinearRegression Object to Find the Best Fit for Each Polynomial Model:               ##
  ##=============================================================================================##

  # Create a LinearRegression object with a forced-origin intercept:

  model = LinearRegression(fit_intercept = False)

  # Loop over polynomial orders from 1 to n_poly:

  for i in range(0, n_poly):

    # Fit the LinearRegression objects to the features and target:

    model.fit(X_list_train[i], y_train)

    # Get the coefficient and bias for the model:

    coef_list.append(np.round(model.coef_[0], 2))
    bias_list.append(np.round(model.intercept_, 2))

    # Get the model's loss:

    model_predictions = model.predict(X_list_test[i])
    loss_list.append(np.sqrt(mean_squared_error(y_test, model_predictions)))

  loss_list_average += np.array(loss_list)

loss_list_average /= n_runs

##=============================================================================================##
## Display The Results:                                                                        ##
##=============================================================================================##

# Create a line plot of Loss vs Polynomial Order

loss_df = pd.DataFrame({"Model": model_list, "Loss": loss_list})

plt.figure(figsize=(10, 6))

sns.lineplot(x = 'Model', y = 'Loss', data=loss_df)

# Add title and labels

plt.title('RMSE Loss vs Model Order')
plt.xlabel('Model')
plt.ylabel('RMSE Loss')

# Set the plot parameters:

plt.grid(True)

y_min = loss_df["Loss"].min() * 0.995
y_max = loss_df["Loss"][1:].max() * 1.005

plt.ylim(y_min, y_max)

# Display the plot

plt.show()

[Return to Top](#Notebook-Start)

<a name="Pipelines"></a>

---

<font size = 6> <b> 8. Introducing Pipelines</b> </font>

---

<font size = 5> <b> 8.1 What is a pipeline?</b> </font>

So we have a process for examining our model by splitting the data, expanding the features, training our model on one portion of the dataset and testing on the other portion of the dataset. While our mehtod is sound, the steps feel
unnecessarily verbose. Luckily, there is a better way. The next approach is oh, so gloriously elegant. Now we are going to learn how to use an sklearn pipeline. A scikit-learn pipeline lets you set up a sequence of useful data processing tasks that will be done sequentially. In the code below is a two-stage pipeline. Each stage of the pipeline has a name, which is just some arbitrary name that you pick as a programmer. If we wish, we can access the individual steps of the pipeline by calling these names.

<br>

* The first stage is a PolynomialFeatures generator which we've named `transform`. You'll observe that this transformer is just our polynomial transformer from before.

* the second stage is a LinearRegression model which we've named `regression`. This stage is just a standard linear regression model.

In [ ]:
##=============================================================================================##
## Create a Data Pipeline:                                                                     ##
##=============================================================================================##

# Create the pipeline:

n_poly = 2

pipelined_model = Pipeline([('transform', PolynomialFeatures(degree = n_poly, include_bias = False)),
                            ('regression', LinearRegression(fit_intercept = False))])


# Display the first stage of the pipeline:

print()
print("Pipeline Stage 1 ('transform'): ")
print()

display(pipelined_model['transform'])

# Display the second stage of the pipeline:

print()
print("Pipeline Stage 2 ('regression'): ")
print()

display(pipelined_model['regression'])

Now we can fit our pipeline using feature and target data.

<br>

* The feature data is passed to the `transform` stage, where the PolynomialFeatures object transformes it into polynomial feature data.

* The polynomial feature data is then passed along with the target data to the second stage where the LinearRegression object is fit.

In [ ]:
##=============================================================================================##
## Fit pipeline on training data:                                                              ##
##=============================================================================================##

pipelined_model.fit(X_train, y_train)

This approach has a few advantages. Most notably, we can apply this model to new data directly. We could just say, pipeline model, please predict the results for the object's position at 0.8 seconds. It will do so without the need to somehow generate the expanded features ourselves. Another nice advantage is that we don't need to explicitly create a new DataFrame of new features when we're training the model. Lastly, we don't have to keep track of separate transformer and regression objects that end up floating around in the variable scope of our Notebooks.

In [ ]:
##=============================================================================================##
## Make the predction of the falling object's position given a specfic time:                   ##
##=============================================================================================##

time     = pd.DataFrame({"Time (s)": [0.8]})

position = pipelined_model.predict(time)[0][0]

# Print the prediction:

print()
print("Predicted Position:", np.round(position, 3), "meters")

Now, we can use the pipeline object to make predictions for our test data.

In [ ]:
##=============================================================================================##
## Predict and Evaluate:                                                                       ##
##=============================================================================================##

y_pred = pipelined_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Root Mean Squared Error on Test Set: {rmse:.3f}")

Now, let's say that we wanted to extract information from a specific step of the pipeline. For example, we might want to get the coefficients and intercept from the regression step. We can access that information by calling the name that we gave that step of the pipeline.

In [ ]:
##=============================================================================================##
## Extract Information From the Pipeline:                                                      ##
##=============================================================================================##

# Get the coefficients from the LinearRegression object:

coef = pipelined_model['regression'].coef_

# Get the intercept from the LinearRegression object:

bias = pipelined_model['regression'].intercept_

# Print the regression model's coefficients and intercept:

print()
print("Model Coefficients:", np.round(coef, 3))
print()
print("Model Intercept:", np.round(bias, 3))

<center>$\rule$</center>

<font size = 5> <b> 8.2 Putting it all together</b> </font>

In [ ]:
##=============================================================================================##
## Prepare the Data For Modeling and Cross-Verification:                                       ##
##=============================================================================================##

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

##=============================================================================================##
## Create a Data Pipeline to Transform the Data and Perform Linear Regression:                 ##
##=============================================================================================##

pipe = Pipeline([('transform',  PolynomialFeatures(degree = 2, include_bias = False)),
                 ('regression', LinearRegression(fit_intercept = False))])

##=============================================================================================##
## Fit the Pipeline to the Training Data:                                                      ##
##=============================================================================================##

pipe.fit(X_train, y_train)

##=============================================================================================##
## Perform Cross-Validation of the Model:                                                      ##
##=============================================================================================##

# Get the model's predictions:

y_pred = pipe.predict(X_test)

# Compute the RMSE Loss:

loss = np.sqrt(mean_squared_error(y_test, y_pred))

print("Root Mean Squared Error on Test Set:", np.round(loss, 3))

##=============================================================================================##
## Extract Information From the Pipeline:                                                      ##
##=============================================================================================##

# Get the coefficients from the LinearRegression object:

coef = pipe['regression'].coef_

# Get the intercept from the LinearRegression object:

bias = pipe['regression'].intercept_

# Print the regression model's coefficients and intercept:

print()
print("Model Coefficients:", np.round(coef, 3))
print()
print("Model Intercept:", np.round(bias, 3))

<center>$\rule$</center>

<font size = 5> <b> 8.3 Using Pipelines to find the best polynomial order</b> </font>

In [ ]:
##=============================================================================================##
## Run the analyis N_Runs number of times and average the results:                             ##
##=============================================================================================##

n_poly = 10

##=============================================================================================##
## Prepare the Data For Modeling and Cross-Verification:                                       ##
##=============================================================================================##

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

##=============================================================================================##
## Create a Data Pipeline to Transform the Data and Perform Linear Regression:                 ##
##=============================================================================================##

pipe_list = []

coef_list = []

loss_list = []

bias_list = []

for i in range(0, n_poly):

  pipe_list.append(
      Pipeline([('transform',  PolynomialFeatures(degree = i + 1, include_bias = False)),
                 ('regression', LinearRegression(fit_intercept = False))])
  )

  ##=============================================================================================##
  ## Fit the Pipeline to the Training Data:                                                      ##
  ##=============================================================================================##

  pipe_list[i].fit(X_train, y_train)

  ##=============================================================================================##
  ## Perform Cross-Validation of the Model:                                                      ##
  ##=============================================================================================##

  # Get the model's predictions:

  y_pred = pipe_list[i].predict(X_test)

  # Compute the RMSE Loss:

  loss = np.sqrt(mean_squared_error(y_test, y_pred))

  ##=============================================================================================##
  ## Extract Information From the Pipeline:                                                      ##
  ##=============================================================================================##

  # Get the coefficients from the LinearRegression object:

  coef = pipe_list[i]['regression'].coef_

  # Get the intercept from the LinearRegression object:

  bias = pipe_list[i]['regression'].intercept_

  # Add values to respective lists:

  loss_list.append(loss)
  coef_list.append(coef[0])
  bias_list.append(bias)

##=============================================================================================##
## Display The Results:                                                                        ##
##=============================================================================================##

# Show the results in DataFrame format:

display_model(model_list, coef_list, bias_list, loss_list, "Polynomial Models", n_items = n_poly,
              trunc = 5)

print('\n\n')

# Create a line plot of Loss vs Polynomial Order

loss_df = pd.DataFrame({"Model": model_list, "Loss": loss_list})

plt.figure(figsize=(10, 6))

sns.lineplot(x = 'Model', y = 'Loss', data=loss_df)

# Add title and labels

plt.title('RMSE Loss vs Model Order')
plt.xlabel('Model')
plt.ylabel('RMSE Loss')

# Set the plot parameters:

plt.grid(True)

y_min = loss_df["Loss"].min() * 0.995
y_max = loss_df["Loss"][1:].max() * 1.005

plt.ylim(y_min, y_max)

# Display the plot

plt.show()

<center>$\rule$</center>

<font size = 5> <b> 8.4 Finding the Best Model</b> </font>

A better way is using the cross_val_score

In [ ]:
from sklearn.model_selection import cross_val_score

##=============================================================================================##
## Run the analyis N_Runs number of times and average the results:                             ##
##=============================================================================================##

n_poly = 10

##=============================================================================================##
## Prepare the Data For Modeling and Cross-Verification:                                       ##
##=============================================================================================##

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

##=============================================================================================##
## Create a Data Pipeline to Transform the Data and Perform Linear Regression:                 ##
##=============================================================================================##

pipe_list = []

coef_list = []

loss_list = []

bias_list = []

cross_val = []

for i in range(0, n_poly):

  pipe_list.append(
      Pipeline([('transform',  PolynomialFeatures(degree = i + 1, include_bias = False)),
                 ('regression', LinearRegression(fit_intercept = False))]),
  )

  score = cross_val_score(pipe_list[i], X, y, cv = 10)


  ##=============================================================================================##
  ## Fit the Pipeline to the Training Data:                                                      ##
  ##=============================================================================================##

  pipe_list[i].fit(X_train, y_train)

  ##=============================================================================================##
  ## Perform Cross-Validation of the Model:                                                      ##
  ##=============================================================================================##

  # Get the model's predictions:

  y_pred = pipe_list[i].predict(X_test)

  # Compute the RMSE Loss:

  loss = np.sqrt(mean_squared_error(y_test, y_pred))

  ##=============================================================================================##
  ## Extract Information From the Pipeline:                                                      ##
  ##=============================================================================================##

  # Get the coefficients from the LinearRegression object:

  coef = pipe_list[i]['regression'].coef_

  # Get the intercept from the LinearRegression object:

  bias = pipe_list[i]['regression'].intercept_

  # Add values to respective lists:

  loss_list.append(loss)
  coef_list.append(coef[0])
  bias_list.append(bias)
  cross_val.append(score.mean())

##=============================================================================================##
## Display The Results:                                                                        ##
##=============================================================================================##

# Show the results in DataFrame format:

display_model(model_list, coef_list, bias_list, loss_list, "Polynomial Models", n_items = n_poly,
              trunc = 5)

print('\n\n')

print("Cross Validation Scores:")

for i in range(0, n_poly):

  print(model_list[i], cross_val[i])

print('\n\n')

# Create a line plot of Loss vs Polynomial Order

loss_df = pd.DataFrame({"Model": model_list, "Loss": loss_list})

plt.figure(figsize=(10, 6))

sns.lineplot(x = 'Model', y = 'Loss', data=loss_df)

# Add title and labels

plt.title('RMSE Loss vs Model Order')
plt.xlabel('Model')
plt.ylabel('RMSE Loss')

# Set the plot parameters:

plt.grid(True)

y_min = loss_df["Loss"].min() * 0.995
y_max = loss_df["Loss"][1:].max() * 1.005

plt.ylim(y_min, y_max)

# Display the plot

plt.show()

[Return to Top](#Notebook-Start)